# Lab 3.2 — Resume Chatbot with Tools & Email

## Introduction

This is the **second lab** in the Resume Chatbot module, building directly on [Lab 3.1](3_lab1.ipynb). You will upgrade your resume chatbot with two capabilities:

1. **Tool calling** — let the LLM invoke Python functions when it needs to take action
2. **Email notifications** — send yourself an email when a visitor shares their contact details or asks a question the bot cannot answer

This is a key step toward a real Agent: the LLM decides *when* to use tools, and your code executes them. You will define tool schemas, implement a dispatcher (`handle_tool_calls`), and run a loop until the LLM produces a final text response.

Before starting, make sure you have:
- Completed Lab 3.1
- `OPENAI_API_KEY`, `YOUR_NAME`, and `YOUR_EMAIL` in your `.env` file
- `MAILER_SEND_API_TOKEN` and `MAILER_SEND_DOMAIN` configured for email (see `.env.example`)
- Your resume at `me/my-resume.md`

---

## Intention (Learning Objectives)

By the end of this lab, you should be able to:

1. **Send email from Python** — use MailerSend to deliver notifications programmatically
2. **Define OpenAI tool schemas** — describe functions the LLM can call using JSON parameter definitions
3. **Implement tool handler functions** — write Python functions that execute when the LLM requests them
4. **Dispatch tool calls** — route incoming `tool_calls` to the correct function and return results to the LLM
5. **Run a tool-calling loop** — keep calling the LLM until `finish_reason` is not `tool_calls`
6. **Integrate tools into a Gradio chatbot** — combine tool calling with the resume chatbot from Lab 3.1

---

## Lab Structure

| Part | Content |
|------|---------|
| **Part 1** | Imports, load environment variables, create OpenAI client |
| **Part 2** | Configure MailerSend and implement `send_email` |
| **Part 3** | Build tool functions — record contact details and unknown questions |
| **Part 4** | Define tool JSON schemas and assemble the `tools` list |
| **Part 5** | Implement `handle_tool_calls` to dispatch tool requests |
| **Part 6** | Load resume, build system prompt, and wire up the tool-calling chat loop |
| **Part 7** | Launch the Gradio chat interface |

> **Note:** Run cells top to bottom in order. Test email sending in Part 2 before moving on.

## Part 1 — Setup

Import dependencies, load your `.env` file, and create the OpenAI client.

In [ ]:
# imports
from mailersend import MailerSendClient, EmailBuilder
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import gradio as gr

In [ ]:
# The usual start

load_dotenv(override=True)
openai = OpenAI()

## Part 2 — Email Setup with MailerSend

This lab uses [MailerSend](https://www.mailersend.com/) to send email notifications from Python. Follow the steps below to get your API token and domain, then add them to your `.env` file.

### Get your MailerSend API token

1. Go to [https://accounts.mailersend.com/](https://accounts.mailersend.com/)
2. **Sign up** for a free account (or log in if you already have one)
3. **Log in** to your MailerSend account
4. Go to [https://app.mailersend.com/domains](https://app.mailersend.com/domains)
5. Click **Manage** on the provided domain
6. Click **Generate new token**, copy the token, and note the domain name

### Add credentials to `.env`

Add the values to your `.env` file (see `.env.example`):

```
MAILER_SEND_API_TOKEN=your_token_here
MAILER_SEND_DOMAIN=your_domain_here
```

- `MAILER_SEND_API_TOKEN` — the token you generated in step 6
- `MAILER_SEND_DOMAIN` — the domain name shown on the Domains page (e.g. `test-xxxxx.mlsender.net`)

Once configured, run the cells below to verify your credentials and implement a helper to send email notifications.

In [ ]:
# For mail sender

mailer_send_api_token = os.getenv("MAILER_SEND_API_TOKEN")
mailer_send_domain = os.getenv("MAILER_SEND_DOMAIN")

if mailer_send_api_token:
    print(f"Mailer Send API Token found and starts with {mailer_send_api_token[0:5]}")
else:
    print("Mailer Send API Token not found")

if mailer_send_domain:
    print(f"Mailer Send Domain found and starts with {mailer_send_domain[0:5]}")
else:
    print("Mailer Send Domain not found")

In [ ]:
# For your data
your_email = os.getenv("YOUR_EMAIL")
your_name = os.getenv("YOUR_NAME")

if your_email:
    print(f"Your Email found and starts with {your_email[0:5]}")
else:
    print("Your Email not found")

if your_name:
    print(f"Your Name found and starts with {your_name[0:5]}")
else:
    print("Your Name not found")

In [ ]:
ms = MailerSendClient(api_key=mailer_send_api_token)

def send_email(sub, message, to_email):
    email = (EmailBuilder()
         .from_email(f"no-reply@{mailer_send_domain}", "No Reply")
         .to_many([{"email": to_email }])
         .subject(sub)
         .text(message)
         .build())

    response = ms.emails.send(email)
    print(f"Email sent: {response.id}")

In [ ]:
send_email("Hello", "World", your_email)

## Part 3 — Tool Functions

Define the Python functions the LLM will call: one to record visitor contact details, and one to log questions it cannot answer.

In [ ]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    send_email("New recording interest", f"Recording interest from {name} with email {email} and notes {notes}", your_email)
    return {"recorded": "ok"}

In [ ]:
def record_unknown_question(question):
    send_email("New unknown question", f"Recording {question} asked that I couldn't answer", your_email)
    return {"recorded": "ok"}

## Part 4 — Tool Schemas

Describe each tool in JSON so the LLM knows when and how to call them. Combine both into a `tools` list passed to the API.

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [ ]:
tools

## Part 5 — Tool Call Dispatcher

When the LLM requests a tool, your code must execute it and return the result. `handle_tool_calls` routes each request to the correct function.

In [ ]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
globals()["record_unknown_question"]("this is a really hard question")

## Part 6 — System Prompt & Tool-Calling Chat Loop

Load your resume from markdown, build an updated system prompt that instructs the LLM to use its tools, and implement the chat loop that runs until the LLM returns a final answer.

In [ ]:
with open("me/my-resume.md", "r", encoding="utf-8") as f:
    resume = f.read()

name = your_name

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Resume:\n{resume}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:

        # This is the call to the LLM - see that we pass in the tools json

        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason
        
        # If the LLM wants to call a tool, we do that!
         
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

## Part 7 — Launch Gradio

Start the chat interface and test tool calling by asking questions the bot cannot answer or by sharing your email.

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

---

## Next Steps

Test both tools in the chat UI: ask something not in your resume (triggers `record_unknown_question`) and share a fake email address (triggers `record_user_details`). Check your inbox for the notification emails. Consider adding a third tool — for example, scheduling a meeting or fetching live data.